# 🎙️ Yeo Discord Voice Bot - All-in-One Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Oreozisbored/yeofiles/blob/main/colab/yeo_all_in_one_colab.ipynb)

Run **both the Discord voice assistant bot and Meta Llama 3.1 8B Instruct together** inside this single Google Colab notebook for on-demand voice channel sessions.

### 📋 Instructions:
1. Ensure your Colab runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. **Cell 1:** Paste your `DISCORD_TOKEN`, configure your settings, and run the cell.
3. **Cell 2:** Run the cell to pull Llama 3.1 8B and start Yeo in Discord!
4. Join a Discord voice channel and type `/join`. Say **"Hey Jarvis"** followed by your request!

In [ ]:
#@title ⚙️ 1. Setup Environment & Configuration
#@markdown ### 🔑 Discord Credentials & Bot Configuration
#@markdown Enter your Discord Bot Token and Bot Owner ID below:
DISCORD_TOKEN = "" #@param {type:"string"}
BOT_OWNER_ID = "" #@param {type:"string"}
BOT_NAME = "Yeo" #@param {type:"string"}

#@markdown ### 🎙️ Wake Word & Model Settings
#@markdown Choose a wake word model (pre-trained "Hey Yeo" & "Hey Verity", or built-in models):
WAKE_WORD = "wakewords/Hey_yee_o.onnx" #@param ["wakewords/Hey_yee_o.onnx", "wakewords/Hey_verity.onnx", "hey_jarvis", "alexa", "weather", "timer"] {type:"string"}
REPO_URL = "https://github.com/Oreozisbored/yeo-dc-bot.git" #@param {type:"string"}
GITHUB_TOKEN = "" #@param {type:"string"}

import os
import shutil
import subprocess

if not DISCORD_TOKEN.strip():
    raise ValueError(
        "❌ DISCORD_TOKEN is required!\n"
        "Please enter your Discord Bot Token in the form field above.\n"
        "Get your token from: Discord Developer Portal -> Applications -> Your Bot -> Bot -> Reset Token."
    )

# Always ensure execution from /content in Google Colab
BASE_DIR = "/content" if os.path.exists("/content") else os.path.abspath(".")
os.chdir(BASE_DIR)
REPO_DIR = os.path.join(BASE_DIR, "yeo-dc-bot")

print("⏳ [1/5] Installing system packages (FFmpeg, libopus, Ollama)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg libopus0 libopus-dev > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("⏳ [2/5] Syncing repository...")
clone_target = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_target = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if not os.path.exists(REPO_DIR):
    !git clone $clone_target "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

# Sync custom sounds and wakewords from public yeofiles CDN safely
print("⏳ Syncing custom wakewords (Hey Yeo, Hey Verity) and soundboard from CDN...")
!rm -rf /tmp/yeofiles
!git clone --depth 1 https://github.com/Oreozisbored/yeofiles.git /tmp/yeofiles > /dev/null 2>&1
!mkdir -p "$REPO_DIR/sounds" "$REPO_DIR/wakewords" "$REPO_DIR/voices"
!cp -rn /tmp/yeofiles/sounds/* "$REPO_DIR/sounds/" 2>/dev/null || true
!cp -rn /tmp/yeofiles/wakewords/* "$REPO_DIR/wakewords/" 2>/dev/null || true

os.chdir(REPO_DIR)

print("⏳ [3/5] Installing Python requirements (NumPy < 2, Discord voice, AI models)...")
!pip install -q --upgrade pip wheel
!pip install -q -r "$REPO_DIR/requirements.txt"

print("⏳ [4/5] Pre-caching default AI models (Piper TTS & openWakeWord)...")
# Pre-cache Piper TTS voice
voice_onnx = os.path.join(REPO_DIR, "voices", "en_US-lessac-medium.onnx")
if not os.path.exists(voice_onnx):
    !wget -q -nc -O "$REPO_DIR/voices/en_US-lessac-medium.onnx" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/lessac/medium/en_US-lessac-medium.onnx" > /dev/null 2>&1 || true
    !wget -q -nc -O "$REPO_DIR/voices/en_US-lessac-medium.onnx.json" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json" > /dev/null 2>&1 || true

# Pre-download openWakeWord models
!python3 -c "import openwakeword; openwakeword.utils.download_models()" > /dev/null 2>&1 || true

print("⏳ [5/5] Writing configuration (.env)...")
env_path = os.path.join(REPO_DIR, ".env")
clean_owner = BOT_OWNER_ID.strip() if BOT_OWNER_ID.strip() else "0"
env_content = f"""DISCORD_TOKEN={DISCORD_TOKEN.strip()}
BOT_OWNER_ID={clean_owner}
BOT_NAME={BOT_NAME.strip() or "Yeo"}
LLM_PROVIDER=openai
LLM_API_BASE=http://localhost:11434/v1
LLM_API_KEY=dummy
LLM_MODEL=llama3.1:8b-instruct
SEARXNG_URL=
WAKE_WORD={WAKE_WORD.strip() or "wakewords/Hey_yee_o.onnx"}
WAKE_WORD_THRESHOLD=0.5
PAST_CONTEXT_SECONDS=60
SILENCE_TIMEOUT_SECONDS=2.0
MAX_COMMAND_SECONDS=15.0
DEFAULT_PIPER_VOICE=en_US-lessac-medium
DEFAULT_TTS_SPEED=1.0
STT_MODEL_SIZE=base.en
STT_DEVICE=cpu
STT_COMPUTE_TYPE=int8
"""
with open(env_path, "w") as f:
    f.write(env_content)

print("\n" + "="*50)
print("🎉 ENVIRONMENT SETUP COMPLETE!")
print(f"🎙️ Configured Wake Word: {WAKE_WORD}")
print("Now click Run on Code Block 2 below to launch Yeo.")
print("="*50)

In [ ]:
#@title 🚀 2. Launch Ollama & Start Yeo Bot
#@markdown Run this cell to start Ollama, pull Llama 3.1 8B, and connect Yeo to Discord.

import os
import subprocess
import time
import urllib.request

# Ensure working directory is the repo directory
REPO_DIR = "/content/yeo-dc-bot" if os.path.exists("/content/yeo-dc-bot") else os.path.abspath("yeo-dc-bot")
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

# 1. Hardware Check
print("🔍 Checking hardware acceleration...")
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"🚀 Detected GPU: {gpu_info}")
except Exception:
    print("⚠️ No NVIDIA GPU detected. Ollama will run in CPU mode.")
    print("💡 For optimal speed, select: Runtime -> Change runtime type -> T4 GPU")

# 2. Check / Start Ollama daemon
def is_ollama_ready():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

if not is_ollama_ready():
    print("⏳ Starting Ollama daemon on localhost:11434...")
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        time.sleep(1)
        if is_ollama_ready():
            break
    if is_ollama_ready():
        print("✅ Ollama daemon started successfully!")
    else:
        print("⚠️ Ollama took longer than expected to report ready, attempting to proceed...")
else:
    print("✅ Ollama daemon is already active on localhost:11434.")

# 3. Pull Llama 3.1 8B Instruct model
MODEL_NAME = "llama3.1:8b-instruct"
print(f"\n⏳ Checking / downloading {MODEL_NAME} (~4.7 GB)...")
!ollama pull $MODEL_NAME
print(f"✅ {MODEL_NAME} is ready!\n")

# 4. Launch Discord Bot
print("="*50)
print("🎙️ YEO DISCORD BOT IS STARTING...")
print("Join your voice channel in Discord and type /join!")
print("="*50 + "\n")

!python3 code.py